# LC 253 — Meeting Rooms II
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Intervals
**Pattern:** Sort by Start + Min-Heap of End Times

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Sort meetings by
start time. Use a min-heap of room end times.
For each new meeting, if the earliest-ending room
is free by then, reuse it — otherwise open a new
room. Heap size is the answer.
</div>

## Official Problem Statement

Given an array of meeting time intervals
`intervals` where `intervals[i] = [starti, endi]`,
return the minimum number of conference rooms
required.

**Example 1:**
```
Input:  intervals = [[0,30],[5,10],[15,20]]
Output: 2
```
**Example 2:**
```
Input:  intervals = [[7,10],[2,4]]
Output: 1
```

**Constraints:**
- `1 <= intervals.length <= 10^4`
- `0 <= starti < endi <= 10^6`

## What This Is Actually Asking

You have a list of meetings each with a start and
end time. Two meetings that overlap need separate
rooms. Find the minimum number of rooms needed
so that no two overlapping meetings share a room.

## Walk Through an Example by Hand

```
intervals = [[0,30],[5,10],[15,20]]

Sort by start: [[0,30],[5,10],[15,20]]

Min-heap tracks end times of rooms in use:

  [0,30]: heap empty -> new room  heap=[30]
          rooms=1

  [5,10]: heap min=30, meeting starts at 5
          5 < 30 -> room not free -> new room
          push 10  heap=[10,30]  rooms=2

  [15,20]: heap min=10, meeting starts at 15
           15 >= 10 -> room free! pop 10 push 20
           heap=[20,30]  rooms still 2

Answer: heap size = 2 = max rooms needed at once

Sanity check on a timeline:
  t=0:  [0,30] starts  Room1 occupied
  t=5:  [5,10] starts  Room1 busy -> Room2
  t=10: [5,10] ends    Room2 free
  t=15: [15,20] starts Room2 free -> reuse Room2
  t=20: [15,20] ends
  t=30: [0,30] ends
  Peak = 2 rooms (t=5 to t=10)
```

## The Picture

```
intervals = [[0,30],[5,10],[15,20]]

Timeline:
  0    5   10   15   20        30
  |----+----+----+----+--------|

  Room1: [0-----------------------------30]
  Room2:      [5--10]  [15--20]

The heap = a waiting list of room end-times.
Smallest end-time = the room that frees up soonest.

For each new meeting (sorted by start):

  heap top < meeting.start?
  YES -> that room is free -> reuse (pop, push new end)
  NO  -> no room free      -> open new room (just push)

heap size after all meetings = rooms needed
```

## When To Use This Pattern

- When asked **how many resources run concurrently**,
  think **sort by start + min-heap of end times**
- When `heap_top <= meeting_start`, think
  **room free — pop old end, push new end**
- When `heap_top > meeting_start`, think
  **room busy — push new end (new room)**
- When the answer is a count of concurrent items,
  think **heap size at the end**

## The Approach

Sort meetings by start time. Create a min-heap to
track end times of rooms currently in use. For each
meeting, if the earliest-ending room finishes at or
before this meeting starts, pop that end time and
reuse the room. Either way, push this meeting's end
time. The final heap size is the number of rooms
required.

In [ ]:
import heapq  # min-heap for earliest-ending room
from typing import List

In [ ]:
def test_harness(func):
    tests = [
        # (intervals, expected)
        ([[0,30],[5,10],[15,20]],   2),
        ([[7,10],[2,4]],            1),
        ([[1,5]],                   1),  # single meeting
        ([[1,2],[2,3],[3,4]],       1),  # sequential
        ([[1,5],[1,5],[1,5]],       3),  # all same time
        ([[1,10],[2,7],[3,19],[8,12],[10,20],[11,30]], 4),
        ([[0,5],[5,10]],            1),  # back-to-back
        ([[1,3],[2,4],[2,5],[3,6]], 3),
    ]

    passed = 0
    for i, (intervals, expected) in enumerate(tests):
        result = func([x[:] for x in intervals])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"intervals={intervals} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def minMeetingRooms(
    intervals: List[List[int]]
) -> int:
    """
    Return minimum meeting rooms required.

    Sort by start. Min-heap tracks end times of
    active rooms. For each meeting: if heap top
    <= start, room is free (pop). Always push this
    meeting's end. Return final heap size.

    Time:  O(n log n) — sort + n heap operations
    Space: O(n) — heap holds at most n end times
    """
    pass


# Quick debug — run this cell while building
print(minMeetingRooms([[0,30],[5,10],[15,20]]))  # 2
print(minMeetingRooms([[7,10],[2,4]]))            # 1
print(minMeetingRooms([[1,5],[1,5],[1,5]]))       # 3
print(minMeetingRooms([[1,2],[2,3],[3,4]]))        # 1

In [ ]:
# Uncomment and run when solution is ready
# test_harness(minMeetingRooms)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — check all pairs | O(n²) | O(1) |
| Two sorted arrays (start/end) | O(n log n) | O(n) |
| Sort + min-heap | O(n log n) | O(n) |

Both optimal approaches cost O(n log n). The heap
approach is more intuitive — it directly simulates
room assignment, making it easier to extend.

## Real World Connection

At Citi, the data platform runs thousands of ETL
jobs on a shared Spark cluster. Each job has a
start and projected end time. The scheduler must
allocate the minimum number of executor pools
so no two overlapping jobs share the same pool.
Meeting Rooms II is the exact model: sort jobs by
start, use a min-heap of pool end times, reuse
freed pools greedily.
On AWS EMR, the cluster auto-scaling logic follows
the same pattern to determine the minimum number
of core nodes required across a set of concurrent
PySpark job windows.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra